# T2M Diagnostic Benchmark — Evaluation Runner

Evaluates one model's **pre-generated, standardised `.npy` motions** with every implemented evaluator. This notebook is shared and normally does **not** change when evaluators are added: it imports all `evaluation/eval_*.py` files automatically and runs them in STEP 10.

| Path | Contents | Owner |
|---|---|---|
| `evaluation/common.py` | settings, input contract, loading, `BaseEvaluator` / registry, Human Gold helpers, unified runner | whole team (PR + review by all) |
| `evaluation/eval_trajectory.py` | Direction — `TrajectoryEvaluator` (+ wrapper for the runner) | member A |
| `evaluation/eval_body_side.py` | Body side — `BodySideEvaluator` | member B |
| `evaluation/eval_<name>.py` | further evaluators (copy `eval_template.py`) | members C, D |
| `evaluation/analysis/analysis_rules.ipynb` | calibration of all evaluators (one shared notebook) | whole team |
| `generation/<model>/` | model runners producing the standardised ZIP | one member per model |
| `benchmark/`, `labels/` | benchmark definition, Human Gold Labels | whole team |

## How to Use This Notebook — Multi-model Pilot Evaluation

This notebook is designed to evaluate multiple Text-to-Motion models using the **same benchmark and evaluator rules**.

### Common workflow for each model

`Standardised .npy`  
→ Load Benchmark Prompts / Requirements  
→ Create or Load Human Gold Labels  
→ Extract Requirement-level Evidence  
→ Apply Automatic Evaluation Rules  
→ Compare with Human Gold

### During the Pilot Stage

During the pilot stage, evaluator rules and thresholds can be developed and validated. Mismatches with Human Gold should be analysed to determine whether a rule needs refinement.

### Final Benchmark

After Cross-model Pilot Validation is completed, the evaluator rules should be **frozen**.

The same frozen rules must then be applied to every model in the final benchmark so that the models are compared under the same evaluation criteria.

# 【Setup】

## STEP 0 — Get the code from GitHub

Clones the repository (first run) or pulls the latest version, then imports `evaluation/common.py` and **every `evaluation/eval_*.py` automatically** — a new evaluator file is picked up without editing this notebook.

- `BRANCH = "main"` for normal use; set it to your branch name to test your work before it is merged.
- Private repository only: add a GitHub token as a Colab secret named `GITHUB_TOKEN` (key icon, left sidebar).
- CPU runtime is enough for evaluation (no GPU needed).

In [ ]:
import importlib, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/Soniaaaa-aa/t2m-capability-benchmark"
BRANCH = "main"                                   # or your feature branch
REPO_DIR = Path("/content/t2m-capability-benchmark")

def _git(*args, cwd=None):
    print("$ git", " ".join(args))
    subprocess.run(["git", *args], cwd=cwd, check=True)

url = REPO_URL
try:  # optional token for a private repository
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    if token:
        url = REPO_URL.replace("https://", f"https://{token}@")
except Exception:
    pass

if not (REPO_DIR / ".git").exists():
    _git("clone", "--branch", BRANCH, url, str(REPO_DIR))
else:
    _git("fetch", "origin", cwd=REPO_DIR)
    _git("checkout", BRANCH, cwd=REPO_DIR)
    _git("pull", "origin", BRANCH, cwd=REPO_DIR)

EVAL_DIR = REPO_DIR / "evaluation"
if str(EVAL_DIR) not in sys.path:
    sys.path.insert(0, str(EVAL_DIR))

import common
importlib.reload(common)                 # pick up changes after a pull
from common import *                     # settings, loaders, registry, gold helpers, runner
evaluator_modules = load_all_evaluators(EVAL_DIR)   # imports every eval_*.py

commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], cwd=REPO_DIR,
                        capture_output=True, text=True).stdout.strip()
print("Framework commit     :", commit or "unknown")

# 【Phase A — Benchmark Input Preparation】

## STEP 3 — Benchmark Input Contract

All input motions must satisfy the same Standardised Motion Contract before evaluation.

Expected format:

- Shape: `[T, 22, 3]`
- Global XYZ coordinates
- `+Y` = Up
- `+Z` = Forward
- `+X` = Right
- Unit = metres
- Frame rate ≈ 20 fps

The benchmark evaluates the contents of the `.npy` motion file, not the file format itself. Model-specific outputs must therefore be converted to this common representation before use.

## STEP 4 — Select the Evaluation Model

Change only `MODEL_NAME`. The motion package must contain `<MODEL_NAME>/<prompt_id>.npy`.
Model-specific generation and conversion happen in `generation/<model>/`.

In [ ]:
# ============================================================
# STEP 4 — Select Model and Standardised Motion Folder
# ============================================================

MODEL_NAME = "MoMADiff"       # ← each member changes only this value

BENCHMARK_INPUT_ROOT = Path("/content/benchmark_inputs")
MODEL_INPUT_DIR = BENCHMARK_INPUT_ROOT / MODEL_NAME

print("=" * 80)
print("MODEL SELECTION")
print("=" * 80)
print("Model       :", MODEL_NAME)
print("Input Folder:", MODEL_INPUT_DIR)

## STEP 3.5 — Import the Pre-generated Motion Package

Upload `<MODEL_NAME>_standardized_pilot.zip`. Skipped if the model folder already exists.

In [ ]:
# ============================================================
# STEP 3.5 — Import Pre-generated Motion Package
# ============================================================

if MODEL_INPUT_DIR.exists():
    print("Motion folder already present — upload skipped:", MODEL_INPUT_DIR)
else:
    from google.colab import files
    print("Upload the standardized motion ZIP for", MODEL_NAME)
    uploaded = files.upload()
    zip_files = [name for name in uploaded if name.lower().endswith(".zip")]
    if len(zip_files) != 1:
        raise RuntimeError(f"Expected exactly 1 ZIP file, found {len(zip_files)}.")
    extract_motion_package(zip_files[0], BENCHMARK_INPUT_ROOT)

if not MODEL_INPUT_DIR.exists():
    raise FileNotFoundError(
        f"Model input folder not found: {MODEL_INPUT_DIR}\n"
        "Check MODEL_NAME and the uploaded motion package."
    )
print("\nSTEP 3.5 — Motion package ready ✅")

## STEP 5 — Load the Benchmark Definition

Read from the repository (`benchmark/`) so every member uses the same version. If the file is not in the repository yet, you are asked to upload it.

In [ ]:
# ============================================================
# STEP 5 — Load Pilot Benchmark Definition
# ============================================================

BENCHMARK_FILE = REPO_DIR / "benchmark" / "pilot_benchmark_definition.json"   # ← file name in the repo

if BENCHMARK_FILE.exists():
    benchmark_definition, pilot_prompts = load_benchmark_definition(BENCHMARK_FILE)
    benchmark_source = str(BENCHMARK_FILE.relative_to(REPO_DIR))
else:
    from google.colab import files
    print(f"{BENCHMARK_FILE.name} is not in the repository — please upload the Pilot Benchmark JSON.")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No file was uploaded.")
    benchmark_source = next(iter(uploaded))
    benchmark_definition, pilot_prompts = parse_benchmark_definition(uploaded[benchmark_source], benchmark_source)

print_benchmark_summary(benchmark_definition, pilot_prompts, benchmark_source)
print("STEP 5 — Pilot Benchmark loaded ✅")

## STEP 6 — Load and Validate Pre-generated `.npy` Motions

Load each Pilot motion from the selected model folder and verify that it satisfies the benchmark input requirements.

The expected file naming convention is based on the Prompt ID, for example:

`C1-01.npy`

`C1-02.npy`

The motion files must already be standardised before this step.

In [ ]:
# ============================================================
# STEP 6 — Load and Validate Pre-generated Motions
# ============================================================

registered_motions, input_results = register_motions(pilot_prompts, MODEL_NAME, MODEL_INPUT_DIR)

# 【Phase B — Automatic Evaluation】

## STEP 7 — Evaluation Case Construction

Construct the evaluation cases by linking each Pilot Prompt and its Atomic Requirements to the corresponding standardised motion file.

Evaluation is performed at the **requirement level**, rather than only assigning one overall score to the complete prompt.

In [ ]:
# ============================================================
# STEP 7 — Evaluation Case Construction
# ============================================================

evaluation_cases = build_evaluation_cases(pilot_prompts, registered_motions, MODEL_NAME)

## STEP 8 — Evaluation Configuration

`EVALUATION_CONFIG` (in `common.py`) maps each Atomic Requirement type to its evaluator; the table shows which evaluators are already implemented.

In [ ]:
used_requirement_types, missing_types, evaluator_status = check_evaluation_config(evaluation_cases)

# STEP 9C — Human Gold Label Template

Create a template for Human Gold Label entry from the Evaluation Cases.

Human Gold Labels are assigned by observing the **motion generated by each model**. Therefore, even for the same prompt, Human Gold may differ between models.

For each Atomic Requirement, the template allows one of the following labels:

- `PASS`
- `FAIL`
- `UNCERTAIN`

In [ ]:
# ============================================================
# STEP 9C — Human Gold Label Template
# ============================================================

human_gold_template = make_gold_template(evaluation_cases)
GOLD_FILE = REPO_DIR / "labels" / gold_label_filename(MODEL_NAME)
print("\nHuman Gold file for this model:", GOLD_FILE.relative_to(REPO_DIR),
      "(exists)" if GOLD_FILE.exists() else "(not created yet)")

# STEP 9D — Human Gold Label Entry (only when creating new labels)

Enter `P` / `F` / `U` for each Atomic Requirement. Skip this and STEP 9E if `labels/<MODEL_NAME>_pilot_human_gold_labels.json` already exists — go to STEP 9F.

In [ ]:
# ============================================================
# STEP 9D — Human Gold Label Entry
# ============================================================

HUMAN_GOLD_LABELS = enter_gold_labels(human_gold_template, MODEL_NAME)

# STEP 9E — Save Human Gold Labels (only when creating new labels)

Saves the labels into the local clone (`labels/`) and downloads a copy. **To share them, add the file to `labels/` on GitHub through a Pull Request.**

In [ ]:
# ============================================================
# STEP 9E — Save Human Gold Labels
# ============================================================

saved = save_gold_labels(HUMAN_GOLD_LABELS, GOLD_FILE)
print("Saved:", saved)
try:
    from google.colab import files
    files.download(str(saved))
except Exception:
    pass

# STEP 9F — Load Saved Human Gold Labels

Loads `labels/<MODEL_NAME>_pilot_human_gold_labels.json` from the repository, or asks you to upload it. **Human Gold is model-specific.**

In [ ]:
# ============================================================
# STEP 9F — Load Saved Human Gold Labels
# ============================================================

if GOLD_FILE.exists():
    gold_source = GOLD_FILE
else:
    from google.colab import files
    print(f"{GOLD_FILE.name} not found in the repository — upload the Human Gold JSON for {MODEL_NAME}.")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No Human Gold JSON was uploaded.")
    gold_source = Path(next(iter(uploaded)))

HUMAN_GOLD_LABELS, label_counts = load_gold_labels(gold_source)

print("\n" + "=" * 80)
print("HUMAN GOLD LABELS LOADED")
print("=" * 80)
print("Model              :", MODEL_NAME)
print("Source             :", gold_source)
print("Pilot Prompts      :", len(HUMAN_GOLD_LABELS))
print("Total Requirements :", label_counts["total"])
print("PASS               :", label_counts["PASS"])
print("FAIL               :", label_counts["FAIL"])
print("UNCERTAIN          :", label_counts["UNCERTAIN"])
print("=" * 80)

## STEP 10 — Run All Evaluators

Every Atomic Requirement is sent to the evaluator mapped in `EVALUATION_CONFIG`, using that evaluator's **current rule / thresholds** (defined in its own `eval_*.py`). Requirement types without an implemented evaluator are listed as `NOT_IMPLEMENTED`; an error in one evaluator is reported as `ERROR` without stopping the others.

The per-evaluator calibration workflows (threshold search, cross-model validation) are in `evaluation/analysis/`.

In [ ]:
# ============================================================
# STEP 10 — Run All Evaluators
# ============================================================

results = run_all_evaluators(evaluation_cases, globals().get("HUMAN_GOLD_LABELS"))

print("=" * 100)
print(f"UNIFIED EVALUATION — {MODEL_NAME}   (framework commit {commit or 'unknown'})")
print("=" * 100)
summary = summarize_results(results)

current_rules = {
    name: {k: v for k, v in vars(cls).items() if k.startswith("CURRENT_")}
    for name, cls in sorted(EVALUATOR_REGISTRY.items())
}
print("\nCurrent evaluator rules:")
for name, rule in current_rules.items():
    print(f"  {name}: {rule}")

mismatches = [r for r in results if r["match"] is False]
errors = [r for r in results if r["status"] == "ERROR"]
if mismatches:
    print("\nMISMATCHES WITH HUMAN GOLD")
    print("-" * 100)
    for r in mismatches:
        print(f"{r['prompt_id']:<7} [{r['requirement_index']}] {r['requirement_type']}={r['expected_value']} "
              f"| auto={r['pass_fail']} human={r['human_label']}")
        print(f"        {r['reason']}")
if errors:
    print("\nERRORS")
    for r in errors:
        print(f"{r['prompt_id']:<7} [{r['requirement_index']}] {r['evaluator']}: {r['reason']}")

## STEP 11 — Save Results

Saves all requirement-level results as JSON (with the framework commit, benchmark version and current evaluator rules) and CSV, then downloads them.

In [ ]:
# ============================================================
# STEP 11 — Save Results
# ============================================================

from datetime import datetime, timezone

metadata = {
    "model": MODEL_NAME,
    "framework_commit": commit,
    "branch": BRANCH,
    "benchmark_source": benchmark_source,
    "benchmark_version": benchmark_definition.get("schema_version"),
    "evaluator_rules": current_rules,
    "created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
}
json_path, csv_path = save_results(results, Path("/content/results") / f"{MODEL_NAME}_pilot_results.json", metadata)
print("Saved:", json_path)
print("Saved:", csv_path)
try:
    from google.colab import files
    files.download(str(json_path))
    files.download(str(csv_path))
except Exception:
    pass

## Adding an evaluator — no change to this notebook

1. Copy `evaluation/eval_template.py` to `evaluation/eval_<name>.py` on your own branch, implement it and uncomment `register_evaluator(...)`.
2. Put your current thresholds in `CURRENT_THRESHOLDS` in that file.
3. Calibration: add a grid for your evaluator in `evaluation/analysis/analysis_rules.ipynb`; optional `tests/test_<name>.py`.
4. Open a Pull Request. After merging, STEP 0 imports your file and STEP 10 runs it automatically.